In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seoultechpse/fenicsx-colab.git"
ROOT = Path("/content")
REPO_DIR = ROOT / "fenicsx-colab"

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

USE_COMPLEX = False  # <--- Set True ONLY if you need complex PETSc
USE_CLEAN = False    # <--- Set True to remove existing environment

opts_str = " ".join(
  [o for c, o in [(USE_COMPLEX, "--complex"), (USE_CLEAN, "--clean")] if c]
)

get_ipython().run_line_magic("run", f"{REPO_DIR / 'setup_fenicsx.py'} {opts_str}")

🔧 FEniCSx Setup Configuration
PETSc type      : real
Clean install   : False

⚠️  Google Drive not mounted — using local cache (/content)

🔧 Installing FEniCSx environment...

🔍 Verifying PETSc type...
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered

✅ FEniCSx setup complete!

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: Real PETSc is installed
   - Recommended for most FEM problems
   - For complex problems, reinstall with --complex


---

In [ ]:
%%fenicsx

"""
Example 1: Basic L² Projection (Corrected)
============================================
This example demonstrates the complete setup for an L² projection problem.
Compatible with current FEniCSx/UFL versions.
"""

import basix.ufl
import ufl

# ============================================================================
# 1. Domain and Element Setup
# ============================================================================

# Define cell type
cell = "triangle"

# Create coordinate element for the mesh (2D, degree 1)
coordinate_element = basix.ufl.element("Lagrange", cell, 1, shape=(2,))

# Define the abstract domain
domain = ufl.Mesh(coordinate_element)

# Define the finite element for the solution space (degree 2)
element = basix.ufl.element("Lagrange", cell, 2)

# ============================================================================
# 2. Function Space
# ============================================================================

# Create the function space V
V = ufl.FunctionSpace(domain, element)

# ============================================================================
# 3. Define Functions
# ============================================================================

# Unknown function u_h ∈ V
uh = ufl.Coefficient(V)

# Define g as a spatial expression
x = ufl.SpatialCoordinate(domain)
g = ufl.sin(x[0]) + ufl.cos(ufl.pi * x[1])

# ============================================================================
# 4. Define the Functional
# ============================================================================

# G(u) = 1/2 ∫_Ω (u-g)² dx
G = 0.5 * (uh - g)**2 * ufl.dx

# ============================================================================
# 5. Compute Variational Forms
# ============================================================================

# Test function δu ∈ V
du = ufl.TestFunction(V)

# Residual: F = dG/du[δu]
F = ufl.derivative(G, uh, du)

# Trial function for Jacobian
dv = ufl.TrialFunction(V)

# Jacobian: J = dF/du[δv]
J = ufl.derivative(F, uh, dv)

# ============================================================================
# 6. Alternative: Direct Definition
# ============================================================================

# We could also define the forms directly:
u_trial = ufl.TrialFunction(V)
v_test = ufl.TestFunction(V)

# Bilinear form: a(u,v) = ∫_Ω u·v dx
a = u_trial * v_test * ufl.dx

# Linear form: L(v) = ∫_Ω g·v dx
L = g * v_test * ufl.dx

# ============================================================================
# 7. Form Analysis
# ============================================================================

# Analyze the Jacobian form
form_data = ufl.algorithms.compute_form_data(
    J,
    do_apply_function_pullbacks=True,
    do_apply_integral_scaling=True,
    do_apply_geometry_lowering=True,
)

print("=" * 70)
print("L² Projection Form Analysis")
print("=" * 70)

# Get form information from arguments
arguments = J.arguments()
print(f"\nNumber of arguments: {len(arguments)}")
print(f"Form type: {'bilinear' if len(arguments) == 2 else 'linear' if len(arguments) == 1 else 'functional'}")
print(f"Arguments: {arguments}")

# Get coefficients
coefficients = J.coefficients()
print(f"\nNumber of coefficients: {len(coefficients)}")
print(f"Coefficients: {coefficients}")

# Check quadrature degree
for integral_data in form_data.integral_data:
    for integral in integral_data.integrals:
        metadata = integral.metadata()
        if 'estimated_polynomial_degree' in metadata:
            print(f"\nEstimated polynomial degree: {metadata['estimated_polynomial_degree']}")

print("\n" + "=" * 70)
print("Alternative Forms (Direct Definition)")
print("=" * 70)

# Check bilinear form
a_args = a.arguments()
print(f"\nBilinear form a(u,v):")
print(f"  Number of arguments: {len(a_args)}")
print(f"  Form type: {'bilinear' if len(a_args) == 2 else 'linear' if len(a_args) == 1 else 'functional'}")

# Check linear form
L_args = L.arguments()
print(f"\nLinear form L(v):")
print(f"  Number of arguments: {len(L_args)}")
print(f"  Form type: {'bilinear' if len(L_args) == 2 else 'linear' if len(L_args) == 1 else 'functional'}")

# ============================================================================
# 8. Additional Form Properties
# ============================================================================

print("\n" + "=" * 70)
print("Additional Form Properties")
print("=" * 70)

# Check integrals
print(f"\nJacobian form J:")
print(f"  Number of integrals: {len(J.integrals())}")
for i, integral in enumerate(J.integrals()):
    print(f"  Integral {i}: type = {integral.integral_type()}, "
          f"subdomain_id = {integral.subdomain_id()}")

print(f"\nBilinear form a:")
print(f"  Number of integrals: {len(a.integrals())}")

print(f"\nLinear form L:")
print(f"  Number of integrals: {len(L.integrals())}")

# ============================================================================
# 9. Verify Equivalence
# ============================================================================

print("\n" + "=" * 70)
print("Verification: Are J and a equivalent?")
print("=" * 70)

# Both should be bilinear forms with the same structure
print(f"\nJ has {len(J.arguments())} arguments")
print(f"a has {len(a.arguments())} arguments")
print("\nNote: J is derived from the functional G, while a is defined directly.")
print("Both represent the same mathematical operation: ∫_Ω u·v dx")

# ============================================================================
# 10. Integrand Analysis
# ============================================================================

print("\n" + "=" * 70)
print("Integrand Structure")
print("=" * 70)

for i, integral in enumerate(J.integrals()):
    print(f"\nJacobian Integral {i}:")
    print(f"  Integrand: {integral.integrand()}")

for i, integral in enumerate(L.integrals()):
    print(f"\nLinear form Integral {i}:")
    print(f"  Integrand: {integral.integrand()}")

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print("""
This example demonstrates:
1. Setting up an abstract domain and finite element
2. Creating function spaces
3. Defining test and trial functions
4. Computing variational forms via symbolic differentiation
5. Alternative direct definition of bilinear and linear forms
6. Analyzing form properties and structure

Key takeaway: UFL provides multiple ways to define the same form,
either through automatic differentiation or direct definition.
""")


L² Projection Form Analysis

Number of arguments: 2
Form type: bilinear
Arguments: (Argument(FunctionSpace(Mesh(blocked element (Basix element (P, triangle, 1, gll_warped, unset, False, float64, []), (2,)), 0), Basix element (P, triangle, 2, gll_warped, unset, False, float64, [])), 0, None), Argument(FunctionSpace(Mesh(blocked element (Basix element (P, triangle, 1, gll_warped, unset, False, float64, []), (2,)), 0), Basix element (P, triangle, 2, gll_warped, unset, False, float64, [])), 1, None))

Number of coefficients: 1
Coefficients: (Coefficient(FunctionSpace(Mesh(blocked element (Basix element (P, triangle, 1, gll_warped, unset, False, float64, []), (2,)), 0), Basix element (P, triangle, 2, gll_warped, unset, False, float64, [])), 0),)

Estimated polynomial degree: 4

Alternative Forms (Direct Definition)

Bilinear form a(u,v):
  Number of arguments: 2
  Form type: bilinear

Linear form L(v):
  Number of arguments: 1
  Form type: linear

Additional Form Properties

Jacobian form J